# NeuroScan — Simple Colab Training

1. **Runtime → Change runtime type → GPU (T4)**
2. On PC: run `scripts/prepare_colab_upload.ps1` and upload `neuroscan_data.zip` to Drive:
   `My Drive / final year project / NeuroScan_Nepal / neuroscan_data.zip`
3. Run all cells below

In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable GPU: Runtime → Change runtime type → GPU'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!pip install -q opencv-python-headless scikit-learn matplotlib

In [ ]:
from google.colab import drive, files
from pathlib import Path
import zipfile

drive.mount('/content/drive')

# Exact path — upload zip here via drive.google.com (NOT empty folders)
ZIP_ON_DRIVE = Path('/content/drive/MyDrive/final year project/NeuroScan_Nepal/neuroscan_data.zip')
WORK = Path('/content/neuroscan')
WORK.mkdir(exist_ok=True)

if ZIP_ON_DRIVE.exists():
    print('Using zip from Drive:', ZIP_ON_DRIVE)
    zip_path = ZIP_ON_DRIVE
else:
    print('Zip not on Drive yet. Upload from PC now (Choose Files):')
    print('File: NeuroScan_Nepal/scripts/neuroscan_data.zip')
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError(
            f'Put neuroscan_data.zip at:\n{ZIP_ON_DRIVE}\n'
            'Create it on PC: scripts/prepare_colab_upload.ps1'
        )
    zip_path = WORK / 'neuroscan_data.zip'
    zip_path.write_bytes(uploaded[next(iter(uploaded))])

with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(WORK)

DATA_ROOT = WORK if (WORK/'normal').exists() else WORK/'raw'
n = len(list((DATA_ROOT/'normal').rglob('*.jpg')))
a = len(list((DATA_ROOT/'abnormal').rglob('*.jpg')))
print(f'Normal: {n}, Abnormal: {a}')
if n == 0 or a == 0:
    raise FileNotFoundError('Zip has no images — re-run prepare_colab_upload.ps1 on PC')

In [ ]:
from google.colab import files
from pathlib import Path

print('Upload neuroscan_colab_train.py from your PC:')
uploaded = files.upload()
Path('neuroscan_colab_train.py').write_bytes(uploaded[next(iter(uploaded))])

In [ ]:
!python neuroscan_colab_train.py --data-root {DATA_ROOT} --out-dir /content/output --batch-size 64 --epochs 30 --lr 0.0005 --patience 7

In [ ]:
from google.colab import files
from pathlib import Path
for f in Path('/content/output').iterdir():
    files.download(str(f))
    print('Downloaded', f.name)